# AI Travel Advisor v3 — Full Notebook

The same AI engine used by the Vercel application. Run this notebook from the project root.

In [ ]:
%pip install -r requirements-dev.txt

## Configure credentials locally

Create `.env` from `.env.example`, then set your own OpenAI Platform key. Never print it.

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your local .env"
print("API key is configured")

## Shared data models and AI engine

`core.py` is the **single source of truth** used by both the notebook and the Vercel endpoint. It includes Pydantic models, context tracking, two-model processing, validation, token metrics, and summarization.

In [ ]:
from core import Advisor, Session, itinerary_markdown
from IPython.display import display, Markdown
session = Session()
advisor = Advisor()

## Interactive notebook conversation

In [ ]:
def ask(message):
    reply, plan = advisor.chat(session, message)
    print("Assistant:", reply)
    if plan:
        display(Markdown(itinerary_markdown(plan)))
    return plan

# Example: ask("I want to spend 5 days in Tunisia enjoying beaches and local food")

## Chat widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
message_box=widgets.Text(placeholder="Tell me about your trip...", description="You:")
send=widgets.Button(description="Send", button_style="primary")
output=widgets.Output()
def send_message(_):
    message=message_box.value.strip()
    if not message: return
    message_box.value=""
    with output:
        print("You:",message)
        try: ask(message)
        except Exception as exc: print("Error:",type(exc).__name__,str(exc)[:250])
send.on_click(send_message)
display(widgets.HBox([message_box,send]),output)

## Usage metrics

In [ ]:
print("Calls:",session.metrics.calls)
print("Input tokens:",session.metrics.input_tokens)
print("Output tokens:",session.metrics.output_tokens)
print("Latency (s):",round(session.metrics.latency_seconds,2))

## Export generated plan

In [ ]:
from pathlib import Path
if session.itinerary:
    Path("itinerary.json").write_text(session.itinerary.model_dump_json(indent=2),encoding="utf-8")
    Path("itinerary.md").write_text(itinerary_markdown(session.itinerary),encoding="utf-8")
    print("Saved itinerary.json and itinerary.md")
else:
    print("Generate an itinerary through the chat first.")

## Deployment

See `README.md` for Vercel steps. Keep `OPENAI_API_KEY` only in Vercel Environment Variables, never in frontend JavaScript.